# Monte Carlo Integration: Statistical Uncertainty and Convergence

Numerically evaluate the integral

$$I=\int_0^1 x^4\,dx=\frac15=0.2$$

using uniform random sampling. We compare numerical estimates with the exact result, examine the distribution of independent estimates, and verify the expected $N^{-1/2}$ statistical scaling.

**Method:** For independent $X_i\sim\mathrm{Uniform}(0,1)$,

$$\hat I_N=\frac1N\sum_{i=1}^N X_i^4.$$

For an integration domain of volume $V$, the general estimator is $\hat I_N=V\,\overline{f(X)}$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

SEED = 2026
rng = np.random.default_rng(SEED)

N_SAMPLES = 1_000
N_REPEATS = 1_000
EXACT_INTEGRAL = 1 / 5

def integrand(x):
    return x**4

def mc_integrate(n_samples, rng):
    """Return the integral estimate and its estimated standard error on [0, 1]."""
    if n_samples < 2:
        raise ValueError("n_samples must be at least 2")
    samples = rng.random(n_samples)
    values = integrand(samples)
    estimate = values.mean()
    standard_error = values.std(ddof=1) / np.sqrt(n_samples)
    return estimate, standard_error


## 1. Single Monte Carlo estimate

The estimated standard error uses the *same samples* as the integral estimate:

$$\widehat{\mathrm{SE}}(\hat I_N)=\frac{s_f}{\sqrt N},\qquad
s_f^2=\frac{1}{N-1}\sum_{i=1}^N\left(f(X_i)-\overline f\right)^2.$$

The theoretical standard error is

$$\mathrm{SE}(\hat I_N)=\sqrt{\frac{\mathrm{Var}(X^4)}{N}}
=\sqrt{\frac{1/9-1/25}{N}}.$$

In [2]:
estimate, estimated_se = mc_integrate(N_SAMPLES, rng)
theoretical_se = np.sqrt((1 / 9 - 1 / 25) / N_SAMPLES)
z_score = (estimate - EXACT_INTEGRAL) / estimated_se

print(f"Monte Carlo estimate: {estimate:.6f}")
print(f"Exact integral:       {EXACT_INTEGRAL:.6f}")
print(f"Estimated std. error: {estimated_se:.6f}")
print(f"Theoretical std. err: {theoretical_se:.6f}")
print(f"Deviation in estimated standard errors: {z_score:.2f}")

assert np.isfinite(estimate) and estimated_se > 0
assert abs(estimate - EXACT_INTEGRAL) < 6 * theoretical_se


Monte Carlo estimate: 0.214751
Exact integral:       0.200000
Estimated std. error: 0.008613
Theoretical std. err: 0.008433
Deviation in estimated standard errors: 1.71


## 2. Distribution of independent estimates

Repeat the complete Monte Carlo experiment 1,000 times. By the central limit theorem, the sampling distribution of the mean approaches a normal distribution as $N$ grows. The curve below uses the **theoretical** normal approximation, rather than an unconstrained histogram fit.

In [3]:
estimates = np.array([mc_integrate(N_SAMPLES, rng)[0] for _ in range(N_REPEATS)])
empirical_mean = estimates.mean()
empirical_sd = estimates.std(ddof=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(estimates, bins=35, density=True, alpha=0.65, label="Independent MC estimates")
x = np.linspace(estimates.min(), estimates.max(), 300)
ax.plot(x, norm.pdf(x, loc=EXACT_INTEGRAL, scale=theoretical_se),
        linewidth=2, label="Theoretical normal approximation")
ax.axvline(EXACT_INTEGRAL, linestyle="--", label="Exact integral = 0.2")
ax.set(xlabel="Monte Carlo estimate", ylabel="Probability density",
       title=f"Sampling distribution ({N_REPEATS} runs, N={N_SAMPLES})")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Mean of estimates:               {empirical_mean:.6f}")
print(f"Empirical SD across experiments: {empirical_sd:.6f}")
print(f"Theoretical SD of estimates:     {theoretical_se:.6f}")
assert abs(empirical_mean - EXACT_INTEGRAL) < 5 * theoretical_se / np.sqrt(N_REPEATS)


Mean of estimates:               0.199780
Empirical SD across experiments: 0.008530
Theoretical SD of estimates:     0.008433


/tmp/ipykernel_436/3184502288.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Convergence and $1/\sqrt{N}$ scaling

For each sample size, generate multiple independent estimates. Compare their empirical standard deviation with the theoretical prediction. The uncertainty decreases as $N^{-1/2}$, even though individual Monte Carlo estimates fluctuate.

In [4]:
sample_sizes = np.array([50, 100, 200, 500, 1_000, 2_000, 5_000, 10_000])
repeats_per_size = 300
empirical_sds = []
mean_absolute_errors = []

for n_samples in sample_sizes:
    runs = np.array([mc_integrate(int(n_samples), rng)[0]
                     for _ in range(repeats_per_size)])
    empirical_sds.append(runs.std(ddof=1))
    mean_absolute_errors.append(np.abs(runs - EXACT_INTEGRAL).mean())

empirical_sds = np.array(empirical_sds)
theoretical_sds = np.sqrt((1 / 9 - 1 / 25) / sample_sizes)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(sample_sizes, empirical_sds, "o-", label="Empirical SD")
ax.loglog(sample_sizes, theoretical_sds, "--", label=r"Theoretical $N^{-1/2}$")
ax.set(xlabel="Samples per estimate (N)", ylabel="Standard deviation of estimates",
       title="Monte Carlo convergence")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()

slope = np.polyfit(np.log(sample_sizes), np.log(empirical_sds), 1)[0]
print(f"Observed log-log slope: {slope:.3f} (expected approximately -0.5)")
assert -0.8 < slope < -0.2


Observed log-log slope: -0.494 (expected approximately -0.5)


/tmp/ipykernel_436/2338934989.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Summary

- Monte Carlo integration estimates an integral using the sample mean of randomly evaluated integrand values.
- Statistical uncertainty is estimated from the **same set of sampled values**, using the sample standard deviation.
- Across repeated independent runs, the estimator's distribution is approximately Gaussian for sufficiently large sample size.
- The standard deviation of estimates decreases approximately as $N^{-1/2}$.

**Limitations:** Samples are independent uniform draws on $[0,1]$; this example does not use importance sampling, Markov-chain Monte Carlo, or variance-reduction techniques.